# ⚡ Flussi di lavoro concorrenti con Microsoft Foundry (Python)

## 📋 Tutorial avanzato sulla elaborazione parallela

Questo notebook dimostra **modelli di flussi di lavoro concorrenti** utilizzando il Microsoft Agent Framework. Imparerai come costruire flussi di lavoro ad alte prestazioni e con elaborazione parallela in cui più agenti AI eseguono contemporaneamente, migliorando drasticamente il throughput e abilitando processi aziendali multi-thread sofisticati.

> **Nota sulla migrazione:** Questo esempio faceva riferimento ai Modelli GitHub. I Modelli GitHub sono deprecati (ritirati a luglio 2026), quindi ora utilizza **Microsoft Foundry** tramite il `FoundryChatClient`, che mira all'API **Responses** di Azure OpenAI.

## 🎯 Obiettivi di apprendimento

### 🚀 **Fondamenti di elaborazione concorrente**
- **Esecuzione parallela degli agenti**: Esegui più agenti simultaneamente per la massima efficienza
- **Orchestrazione dei flussi di lavoro**: Coordina operazioni concorrenti mantenendo la coerenza dei dati
- **Ottimizzazione delle prestazioni**: Ottieni un significativo aumento di velocità tramite l'elaborazione parallela
- **Gestione delle risorse**: Utilizza efficientemente le risorse dei modelli AI per operazioni concorrenti

### 🏗️ **Modelli avanzati di concorrenza**
- **Elaborazione fork-join**: Dividi il lavoro tra più agenti e unisci i risultati
- **Parallelismo a pipeline**: Sovrapposizione delle fasi di esecuzione per un throughput continuo
- **Bilanciamento del carico**: Distribuisci equamente il lavoro tra le risorse agenti disponibili
- **Punti di sincronizzazione**: Coordina gli agenti concorrenti in fasi critiche del flusso di lavoro

### 🏢 **Applicazioni concorrenti per l'impresa**
- **Elaborazione di documenti ad alto volume**: Elabora più documenti contemporaneamente
- **Analisi dei contenuti in tempo reale**: Analisi concorrente di flussi di dati in ingresso
- **Ottimizzazione dell'elaborazione batch**: Massimizza il throughput per operazioni su larga scala
- **Analisi multimodale**: Elaborazione parallela di diversi tipi di contenuto (testo, immagini, dati)

## ⚙️ Prerequisiti e configurazione

### 📦 **Dipendenze richieste**

Installa Agent Framework con capacità di flussi di lavoro concorrenti:

```bash
pip install agent-framework -U
```

### 🔑 **Configurazione Microsoft Foundry**

Accedi con Azure CLI (`az login`) così `AzureCliCredential` può autenticarti, poi imposta i dettagli del tuo progetto Microsoft Foundry.

**Configurazione ambiente (file .env):**
```env
AZURE_AI_PROJECT_ENDPOINT=https://<your-project>.services.ai.azure.com
AZURE_AI_MODEL_DEPLOYMENT_NAME=gpt-5-mini
```

**Considerazioni sull'elaborazione concorrente:**
- **Limiti di velocità**: Monitora i limiti di Azure OpenAI per richieste concorrenti
- **Uso delle risorse**: Considera l'uso di memoria e CPU con più agenti concorrenti
- **Gestione degli errori**: Implementa un recupero robusto per operazioni parallele

### 🏗️ **Architettura dei flussi di lavoro concorrenti**

```mermaid
graph TD
    A[Inizio del Flusso di Lavoro] --> B[Esecuzione Concurrente]
    B --> C[Pool di Agenti 1]
    B --> D[Pool di Agenti 2]
    B --> E[Pool di Agenti 3]
    C --> F[Aggregazione dei Risultati]
    D --> F
    E --> F
    F --> G[Output Finale]
    
    H[Microsoft Foundry] --> C
    H --> D
    H --> E
```

**Vantaggi chiave:**
- **⚡ Prestazioni**: Significativo aumento di velocità tramite esecuzione parallela
- **📈 Scalabilità**: Gestisci carichi di lavoro aumentati senza incremento proporzionale del tempo
- **🔄 Efficienza**: Migliore utilizzo delle risorse computazionali disponibili
- **🎯 Throughput**: Elabora più lavoro nello stesso intervallo di tempo

## 🎨 **Modelli di progettazione per flussi di lavoro concorrenti**

### 🔍 **Pipeline di ricerca e analisi**
```
Research Task → Parallel Research Agents → Content Synthesis → Quality Review
```

### 📊 **Flusso di lavoro per l'elaborazione dati**
```
Input Data → Concurrent Processing Agents → Result Aggregation → Final Report
```

### 🎭 **Pipeline di creazione dei contenuti**
```
Content Brief → Parallel Content Generators → Review & Merge → Final Content
```

### 🔄 **Elaborazione a più fasi**
```
Input → Stage 1 (Concurrent) → Stage 2 (Concurrent) → Stage 3 (Sequential) → Output
```

## 🏢 **Benefici delle prestazioni a livello aziendale**

### ⚡ **Ottimizzazione del throughput**
- **Esecuzione parallela**: Molti agenti che lavorano simultaneamente
- **Utilizzo delle risorse**: Massima efficienza della capacità del modello AI disponibile
- **Riduzione dei tempi**: Diminuzione significativa del tempo totale di elaborazione
- **Architettura scalabile**: Aggiungi facilmente più agenti concorrenti secondo necessità

### 🛡️ **Affidabilità e resilienza**
- **Tolleranza agli errori**: Il fallimento di un agente non blocca l'intero flusso di lavoro
- **Isolamento degli errori**: Problemi in un ramo concorrente non influenzano gli altri
- **Degradazione controllata**: Il sistema continua a funzionare anche con capacità agenti ridotta
- **Meccanismi di recupero**: Riprova automatica e gestione errori per operazioni fallite

### 📊 **Monitoraggio e osservabilità**
- **Tracciamento dell'esecuzione concorrente**: Monitora il progresso di tutte le operazioni parallele
- **Metriche di prestazioni**: Misura l'aumento della velocità e i guadagni di efficienza
- **Analisi uso delle risorse**: Ottimizza l'allocazione degli agenti concorrenti
- **Identificazione dei colli di bottiglia**: Trova e risolvi i vincoli delle prestazioni

Costruiamo flussi di lavoro AI concorrenti ad alte prestazioni! 🚀


In [ ]:
# Already covered by repo-level requirements.txt; left for reference.
# !pip install agent-framework -U

In [ ]:
import os
from typing import Any

from agent_framework import (
    Executor,
    Message,
    WorkflowBuilder,
    WorkflowContext,
    WorkflowViz,
    handler,
)
from agent_framework.foundry import FoundryChatClient
from azure.identity import AzureCliCredential
from dotenv import load_dotenv

load_dotenv()


In [ ]:
# Configure the Microsoft Foundry client with keyless authentication.
# FoundryChatClient targets the Azure OpenAI Responses API.
provider = FoundryChatClient(
    project_endpoint=os.environ["AZURE_AI_PROJECT_ENDPOINT"],
    model=os.environ["AZURE_AI_MODEL_DEPLOYMENT_NAME"],
    credential=AzureCliCredential(),
)


In [ ]:
ResearcherAgentName = "Researcher-Agent"
ResearcherAgentInstructions = "You are my travel researcher, working with me to analyze the destination, list relevant attractions, and make detailed plans for each attraction."

In [ ]:
PlanAgentName = "Plan-Agent"
PlanAgentInstructions = "You are my travel planner, working with me to create a detailed travel plan based on the researcher's findings."

In [ ]:
research_agent = provider.as_agent(
    name=ResearcherAgentName,
    instructions=ResearcherAgentInstructions,
)

plan_agent = provider.as_agent(
    name=PlanAgentName,
    instructions=PlanAgentInstructions,
)


In [ ]:
# A passthrough executor that broadcasts the user input to every agent in parallel.
class InputDispatcher(Executor):
    """Forward the user input unchanged to all participating agents."""

    @handler
    async def forward(self, text: str, ctx: WorkflowContext[str]) -> None:
        await ctx.send_message(text)


dispatcher = InputDispatcher(id="dispatcher")
agents = [research_agent, plan_agent]

workflow = (
    WorkflowBuilder(
        start_executor=dispatcher,
        output_executors=agents,
    )
    .add_fan_out_edges(dispatcher, agents)
    .build()
)

In [ ]:
print("Generating workflow visualization...")
viz = WorkflowViz(workflow)
# Print out the mermaid string.
print("Mermaid string: \n=======")
print(viz.to_mermaid())
print("=======")
# Print out the DiGraph string.
print("DiGraph string: \n=======")
print(viz.to_digraph())
print("=======")
# SVG export needs the optional graphviz extra plus the graphviz system binary;
# fall back gracefully if it is not available.
try:
    svg_file = viz.export(format="svg")
    print(f"SVG file saved to: {svg_file}")
except ImportError as e:
    svg_file = None
    print(f"SVG export skipped (install graphviz to enable): {e}")

In [ ]:
from IPython.display import SVG, display, HTML
import os

print(f"Attempting to display SVG file at: {svg_file}")

if svg_file and os.path.exists(svg_file):
    try:
        # Preferred: direct SVG rendering
        display(SVG(filename=svg_file))
    except Exception as e:
        print(f"⚠️ Direct SVG render failed: {e}. Falling back to raw HTML.")
        try:
            with open(svg_file, "r", encoding="utf-8") as f:
                svg_text = f.read()
            display(HTML(svg_text))
        except Exception as inner:
            print(f"❌ Fallback HTML render also failed: {inner}")
else:
    print("❌ SVG file not found. Ensure viz.export(format='svg') ran successfully.")

In [ ]:
events = await workflow.run("Plan a trip to Seattle in December")
outputs = events.get_outputs()

In [ ]:
if outputs:
    print("===== Final Aggregated Responses =====")
    # outputs is a list of AgentResponse objects, one per output executor
    # (research_agent then plan_agent), in the order given to output_executors.
    for i, response in enumerate(outputs, start=1):
        print(f"{'-' * 60}\n\n{i:02d}:\n{response.text}")

---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Disclaimer**:
Questo documento è stato tradotto utilizzando il servizio di traduzione AI [Co-op Translator](https://github.com/Azure/co-op-translator). Sebbene ci impegniamo per garantire la precisione, si prega di notare che le traduzioni automatizzate possono contenere errori o imprecisioni. Il documento originale nella sua lingua nativa deve essere considerato la fonte autorevole. Per informazioni critiche, si raccomanda una traduzione professionale effettuata da un essere umano. Non siamo responsabili per eventuali malintesi o interpretazioni errate derivanti dall’uso di questa traduzione.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
